# SaaS Retention Deep Dive

**Full-Funnel Retention & Churn Prediction Engine (LTV:CAC Optimizer)**

This notebook is the executive walkthrough of the dual-door framework:

| Door | Question | Primary tables |
| --- | --- | --- |
| **Front Door** | Are we buying the *right* customers? | `acquisition_leads.csv` |
| **Back Door** | Are we keeping the customers we already paid for? | `user_telemetry_churn.csv` |

Parts:
1. Exploratory analysis of CAC vs. LTV by channel and engagement distributions
2. Hypothesis testing (Welch t-test + chi-square) with interpretations
3. ROC and Precision-Recall curves: Logistic Regression vs. XGBoost
4. Early-warning scoring and Customer Success playbook

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.generate_data import generate_datasets
from src.data_pipeline import run_pipeline
from src.stats_engine import print_statistical_summary, run_hypothesis_tests
from src.churn_model import ChurnScoringEngine
from src.paths import ACQUISITION_LEADS_PATH

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 14

if not ACQUISITION_LEADS_PATH.exists():
    generate_datasets()

df = run_pipeline()
df.head()

## Part 1 — Front Door vs. Back Door EDA

The Front Door is an *economics* problem: CAC paid today versus estimated lifetime value.  
The Back Door is a *behavior* problem: logins, feature adoption, inactivity, and support load.

Healthy B2B SaaS targets an LTV:CAC ratio above **3:1**. Ratios below 1 mean the book of business is destroying capital even before churn is modeled.

In [ ]:
print(f"Customers: {len(df):,}")
print(f"Churn rate: {df['churned'].mean():.1%}")
print(f"Median LTV:CAC: {df['ltv_cac_ratio'].median():.2f}")
print(f"High-risk inactivity (>14 days): {df['high_risk_inactivity'].mean():.1%}")

channel_econ = (
    df.groupby("acquisition_channel")
    .agg(
        n=("customer_id", "count"),
        median_cac=("cac_usd", "median"),
        median_ltv=("ltv_est", "median"),
        median_ltv_cac=("ltv_cac_ratio", "median"),
        churn_rate=("churned", "mean"),
        mean_adoption=("feature_adoption_score", "mean"),
    )
    .sort_values("median_ltv_cac", ascending=False)
)
channel_econ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(
    data=df.sample(1200, random_state=42),
    x="cac_usd",
    y="ltv_est",
    hue="acquisition_channel",
    alpha=0.55,
    ax=axes[0],
    s=28,
)
axes[0].set_title("Front Door: CAC vs. Estimated LTV by Channel")
axes[0].set_xlabel("CAC (USD)")
axes[0].set_ylabel("Estimated LTV (USD)")

order = (
    df.groupby("acquisition_channel")["ltv_cac_ratio"]
    .median()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=df,
    x="acquisition_channel",
    y="ltv_cac_ratio",
    order=order,
    ax=axes[1],
)
axes[1].axhline(3.0, color="crimson", ls="--", lw=1.4, label="3:1 efficiency line")
axes[1].set_title("LTV:CAC Distribution by Acquisition Channel")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

sns.histplot(
    data=df, x="feature_adoption_score", hue="churned",
    bins=30, kde=True, ax=axes[0, 0], palette="Set1", stat="density", common_norm=False,
)
axes[0, 0].set_title("Back Door: Feature Adoption by Churn Status")

sns.histplot(
    data=df, x="avg_weekly_logins", hue="churned",
    bins=30, kde=True, ax=axes[0, 1], palette="Set1", stat="density", common_norm=False,
)
axes[0, 1].set_title("Average Weekly Logins by Churn Status")

sns.boxplot(
    data=df, x="churned", y="days_since_last_login", ax=axes[1, 0], palette="Set1",
)
axes[1, 0].axhline(14, color="black", ls="--", lw=1.2)
axes[1, 0].set_title("Days Since Last Login (14-day inactivity threshold)")

sns.kdeplot(
    data=df, x="engagement_index", hue="contract_type", fill=True, ax=axes[1, 1],
)
axes[1, 1].set_title("Engagement Index by Contract Type")
fig.tight_layout()
plt.show()

## Part 2 — Hypothesis Testing

Google Advanced Data Analytics framing:

1. **Two-sample Welch t-test** — Is mean `feature_adoption_score` different for churned vs. retained accounts?
2. **Chi-square test of independence** — Is `churned` statistically associated with `acquisition_channel`?

We report p-values, degrees of freedom, Cohen's *d*, and Cramér's V so significance is never confused with effect size.

In [ ]:
results = run_hypothesis_tests(df)
print_statistical_summary(results)

ttest = results["ttest"]
chi = results["chi_square"]

print("Annotated interpretation")
print(
    f"Adoption gap = {ttest.mean_b - ttest.mean_a:.2f} points "
    f"(retained minus churned). Cohen's d = {ttest.cohens_d:.2f}."
)
if ttest.significant:
    print(
        "Reject H0. Product activation is not cosmetic: churned customers sit on a "
        "materially different adoption distribution. Prioritize time-to-value SLAs "
        "in the first 14 days."
    )
if chi.significant:
    print(
        "Reject H0 for channel independence. Front Door mix (especially outbound "
        "cold email vs. partner / organic) leaks into Back Door churn. Treat CAC "
        "reallocation as a retention program, not only a growth program."
    )

## Part 3 — Predictive Models: Logistic Regression vs. XGBoost

Training protocol:
- Stratified 80/20 split on `churned`
- `StandardScaler` on the logistic design matrix; VIF pruning at 10
- XGBoost (RandomForest fallback) with `GridSearchCV` refit on PR-AUC

PR-AUC is the primary ranking metric because churn is the minority class. ROC-AUC is reported for comparability.

In [ ]:
engine = ChurnScoringEngine()
engine.fit(df, tune=True)

rows = []
for name, report in engine.reports_.items():
    rows.append(
        {
            "model": name,
            "accuracy": report.accuracy,
            "precision": report.precision,
            "recall": report.recall,
            "f1": report.f1,
            "roc_auc": report.roc_auc,
            "pr_auc": report.pr_auc,
            "brier": report.brier,
        }
    )
pd.DataFrame(rows).set_index("model").round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, report in engine.reports_.items():
    fpr, tpr, _ = report.extras["roc_curve"]
    prec, rec, _ = report.extras["pr_curve"]
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={report.roc_auc:.3f})")
    axes[1].plot(rec, prec, lw=2, label=f"{name} (PR-AUC={report.pr_auc:.3f})")

axes[0].plot([0, 1], [0, 1], ls="--", c="gray", lw=1, label="Chance")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves")
axes[0].legend(loc="lower right")

baseline = df["churned"].mean()
axes[1].axhline(baseline, ls="--", c="gray", lw=1, label=f"Prevalence={baseline:.2f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves")
axes[1].legend(loc="lower left")
fig.tight_layout()
plt.show()

if engine.feature_importances_ is not None:
    top = engine.feature_importances_.head(12)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=top, y="feature", x="importance", ax=ax, color="#2c7fb8")
    ax.set_title("Production Model Feature Importance")
    fig.tight_layout()
    plt.show()

## Part 4 — Early-Warning Scoring & Customer Success Workflow

Active (not yet churned) accounts with predicted risk ≥ **0.65** are written to `data/processed/churn_risk_alerts.csv`.

Intervention policy is rule-based on top of the model score so CS managers get an *action*, not only a probability:

| Trigger | Recommended motion |
| --- | --- |
| >21 days dark | 48-hour CSM sprint + executive sponsor |
| Adoption < 4 / 10 | Core-feature activation workshop |
| ≥4 support tickets | Technical account review |
| Monthly contract + strong LTV:CAC | Annual conversion offer |
| LTV:CAC < 3 | Value review; do not buy retention with unstructured discounting |

In [ ]:
alerts = engine.score_active_customers(df, threshold=0.65)
print(f"Alerts generated: {len(alerts):,}")
display(alerts.head(15))

if not alerts.empty:
    print("\nIntervention mix")
    print(alerts["recommended_intervention"].value_counts().to_string())
    print("\nRisk-tier mix")
    print(alerts["risk_tier"].value_counts().to_string())

### Operating cadence (recommended)

1. **Daily** — refresh telemetry, rescore the book, push Critical-tier alerts to the CSM queue.
2. **Weekly** — Front Door review: pause or recut outbound sequences whose LTV:CAC and churn both underperform organic/partner.
3. **Monthly** — retrain the production model; audit calibration (Brier score) and PR-AUC on a rolling window.
4. **Quarterly** — reset the 3:1 LTV:CAC hurdle by segment and feed Finance a channel-level contribution margin view.

Onboarding friction (low `feature_adoption_score` among churners) and channel quality (outbound vs. partner) are the two highest-leverage findings in this simulation. Fixing either door in isolation leaves money on the table; the engine is designed so Growth and Customer Success share one scored ledger.